In [ ]:
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog

# Ocultar la ventana principal de tkinter
root = tk.Tk()
root.withdraw()

# Paso 1: Seleccionar el archivo de ventas
nombre_archivo_ventas = filedialog.askopenfilename(
    title="Selecciona el archivo de ventas (.xlsx)",
    filetypes=[("Archivos de Excel", "*.xlsx")]
)

# Verificar si se seleccionó un archivo
if not nombre_archivo_ventas:
    print("No se seleccionó el archivo de ventas.")
    exit()

# Paso 2: Cargar el archivo de ventas
try:
    df_ventas = pd.read_excel(nombre_archivo_ventas)
    print(f"Archivo de ventas '{os.path.basename(nombre_archivo_ventas)}' cargado correctamente.")
except Exception as e:
    print(f"Error al cargar el archivo de ventas: {e}")
    exit()

# Paso 3: Seleccionar el archivo de notas crédito
nombre_archivo_notas_credito = filedialog.askopenfilename(
    title="Selecciona el archivo de notas crédito (.xlsx)",
    filetypes=[("Archivos de Excel", "*.xlsx")]
)

# Verificar si se seleccionó un archivo
if not nombre_archivo_notas_credito:
    print("No se seleccionó el archivo de notas crédito.")
    exit()

# Paso 4: Cargar el archivo de notas crédito
try:
    df_notas_credito = pd.read_excel(nombre_archivo_notas_credito)
    print(f"Archivo de notas crédito '{os.path.basename(nombre_archivo_notas_credito)}' cargado correctamente.")
except Exception as e:
    print(f"Error al cargar el archivo de notas crédito: {e}")
    exit()

# Paso 5: Extraer el número de factura de la columna "Referencia" en las notas crédito
df_notas_credito['NUMERO_FACTURA'] = df_notas_credito['Líneas de factura/Referencia'].apply(
    lambda x: re.search(r'(FEVY\d+)', x).group(1) if pd.notna(x) and re.search(r'(FEVY\d+)', x) else None
)
df_notas_credito = df_notas_credito.drop(columns=['Líneas de factura/Número'])
df_notas_credito = df_notas_credito.rename(columns={'NUMERO_FACTURA': 'Líneas de factura/Número'})
# Paso 6: Convertir las cantidades y totales de las notas crédito a valores negativos
df_notas_credito['Líneas de factura/Cantidad'] = -df_notas_credito['Líneas de factura/Cantidad']
df_notas_credito['Líneas de factura/Total'] = -df_notas_credito['Líneas de factura/Total']
# Paso 8: Crear una columna temporal que combine NUMERO_FACTURA y PRODUCTO
df_ventas['NUMERO_FACTURA-PRODUCTO'] = df_ventas['Líneas de factura/Número'] + '-' + df_ventas['Líneas de factura/Producto']
df_notas_credito['NUMERO_FACTURA-PRODUCTO'] = df_notas_credito['Líneas de factura/Número'] + '-' + df_notas_credito['Líneas de factura/Producto']
# Paso 9: Filtrar las notas crédito para incluir solo las que coinciden con ventas existentes
notas_credito_validas = df_notas_credito['NUMERO_FACTURA-PRODUCTO'].isin(df_ventas['NUMERO_FACTURA-PRODUCTO'])
df_notas_credito_filtrado = df_notas_credito[notas_credito_validas]
# Paso 8: Combinar ambos datasets (ventas y notas crédito)
df_consolidado = pd.concat([df_ventas, df_notas_credito_filtrado], ignore_index=True)
# Paso 10: Agrupar por la columna temporal NUMERO_FACTURA-PRODUCTO
df_consolidado = df_consolidado.groupby(
    'NUMERO_FACTURA-PRODUCTO',  # Agrupar por la combinación de factura y producto
    as_index=False
).agg({
    'Líneas de factura/Fecha de factura': 'first',
    'Líneas de factura/Asociado': 'first',
    'Líneas de factura/Número': 'first',
    'Líneas de factura/Producto': 'first',
    'Líneas de factura/Cantidad': 'sum',  # Sumar las cantidades
    'Líneas de factura/Total': 'sum',     # Sumar los totales
    'Líneas de factura/Moneda/Tasa actual': 'first',
    'Líneas de factura/Asociado/Número de Identificación': 'first',
    'Líneas de factura/Asociado/Teléfono': 'first',
    'Líneas de factura/Asociado/Correo electrónico': 'first',
    'Líneas de factura/Asociado/Ciudad': 'first',
    'Líneas de factura/Asociado/Estado': 'first',
    'Equipo de Ventas': 'first',
    'Líneas de factura/Referencia': 'first',
    'Asesor Comercial': 'first',
    'Origen': 'first',
    'Origen/Nombre de la Fuente': 'first'

})
# Paso 12: Eliminar la columna temporal NUMERO_FACTURA-PRODUCTO
df_consolidado.drop(columns=['NUMERO_FACTURA-PRODUCTO'], inplace=True)
# Paso 13: Filtrar solo las filas donde la cantidad sea mayor que 0 (eliminar ventas canceladas)
df_consolidado = df_consolidado[df_consolidado['Líneas de factura/Cantidad'] > 0]
# Paso 14: Generar el nombre del archivo de salida
nombre_archivo_salida = os.path.splitext(nombre_archivo_ventas)[0] + "_procesado.xlsx"

# Paso 15: Guardar el archivo consolidado
try:
    df_consolidado.to_excel(nombre_archivo_salida, index=False)
    print(f"Archivo consolidado guardado como '{nombre_archivo_salida}'.")
except Exception as e:
    print(f"Error al guardar el archivo consolidado: {e}")

# Paso 16: Obtener el listado de facturas afectadas por notas crédito
facturas_afectadas = df_notas_credito_filtrado[['Líneas de factura/Número', 'Líneas de factura/Producto', 'Líneas de factura/Cantidad', 'Líneas de factura/Total']].dropna(subset=['Líneas de factura/Número'])

# Paso 17: Mostrar el listado de facturas afectadas
print("\nFacturas afectadas por notas crédito:")
print(facturas_afectadas.shape)

# Paso 18: Guardar el listado de facturas afectadas en un archivo Excel (opcional)
nombre_archivo_facturas_afectadas = os.path.splitext(nombre_archivo_ventas)[0] + "_facturas_afectadas.xlsx"
try:
    facturas_afectadas.to_excel(nombre_archivo_facturas_afectadas, index=False)
    print(f"Listado de facturas afectadas guardado como '{nombre_archivo_facturas_afectadas}'.")
except Exception as e:
    print(f"Error al guardar el listado de facturas afectadas: {e}")

Archivo de ventas 'agosto.xlsx' cargado correctamente.
Archivo de notas crédito 'notas_agosto.xlsx' cargado correctamente.
Archivo consolidado guardado como 'C:/Users/Dataa/Desktop/VENTAS/VENTA MENSUAL/agosto_procesado.xlsx'.

Facturas afectadas por notas crédito:
    Líneas de factura/Número                    Líneas de factura/Producto  \
1                  FEVY64237                      [TNGL01] SHAMPOO TONGOLÉ   
2                  FEVY64237                  [TNGL02] TRATAMIENTO TONGOLÉ   
3                  FEVY64237                     [TNGL03] LEAVE ON TONGOLÉ   
4                  FEVY64237                   [TNGL04] MASCARILLA TONGOLÉ   
5                  FEVY64240                     [PCN02] SHAMPOO LA POCION   
..                       ...                                           ...   
216                FEVY62238              [PCN19] DUTONIC (TONICO CAPILAR)   
217                FEVY62238          [PCN25] SHAMPOO CONTROL GRASA POCION   
229                FEVY61891     

In [4]:
facturas_afectadas.shape

(188, 4)